# Section 3: Build, Train, or Estimate the ML Model --> Chapter 11. Ranking

---
---

**Book: Applied Machine Learning for Data Science Practitioners**
<BR>

**Author:** Vidya Subramanian (https://www.linkedin.com/in/vidyas/)

**Note:** Code is only included here for the steps that require it. Please refer to the book for the complete set of steps related to the goals covered in the chapter.

---
---

---

☑ **Install libraries**

---

In [ ]:
# In case you need to install the relevent packages, please uncomment lines below and run (once only)

!pip install scikit-learn==1.2.2
!pip install pandas==2.0.3
!pip install numpy==1.25.2
!pip install matplotlib==3.7.1
!pip install pydrive2==1.6.3
!pip install IPython==7.34.0
!pip install seaborn==0.13.1
!pip install scipy==1.11.4
!pip install google.colab
!pip install pydrive2==1.6.3
!pip install oauth2client==4.1.3
!pip install xgboost==2.0.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.2 MB/s eta 0:00:00


---

☑ **Import libraries**

---

In [ ]:
# Set some common environment variables and Imports
import numpy as np
import pandas as pd  # Import pandas module for data manipulation
import warnings  # Warnings
from IPython.core.display import display, HTML  # Make the Jupyter chunk window wider
import sys
import time

from sklearn.feature_selection import *
from sklearn.manifold import spectral_embedding
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import *

import scipy
from scipy.stats import pearsonr, kendalltau, spearmanr
import scipy.stats as stats
from itertools import combinations

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)  # Setting to display all columns in a single row
display(HTML("<style>.container { width:100% !important; }</style>")) # Make the Jupyter chunk window wider

from pydrive2.auth import GoogleAuth  # Import GoogleAuth class from the pydrive2.auth module
from pydrive2.drive import GoogleDrive  # Import GoogleDrive class from the pydrive2.drive module
from google.colab import auth  # Import auth class from the google.colab module
from oauth2client.client import GoogleCredentials  # Import GoogleCredentials class from the oauth2client.client module

## 3.1 Pointwise Ranking

In [ ]:
import time
from google.colab import auth
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import ndcg_score
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

# Function to help display
def print_pretty_header(title, subtitle):
    # Define header formatting
    line_length = 50
    header_padding = 2
    # Calculate the width for the title text
    title_width = line_length * header_padding
    print("#" * title_width + "\n")
    print(title.center(title_width) + "\n")
    if subtitle:
        # Calculate the width for the subtitle text
        subtitle_width = line_length - 2 * header_padding
        print(subtitle.center(title_width) + "\n")
    print("#" * title_width + "\n")

# Function to retrieve training data
def _1_get_data():
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    drive = GoogleDrive(gauth)

    colab_file_name = 'S3_Ch11_Ranking.csv'
    data_link = 'https://drive.google.com/open?id=1Tyl8Hpbqg2quiz0Bk2G39ejusFFZKfCF'
    ignore_str, file_id = data_link.split('=')
    downloaded = drive.CreateFile({'id': file_id})
    downloaded.GetContentFile(colab_file_name)
    df_raw_data = pd.read_csv(colab_file_name)
    pd.set_option('display.max_columns', None)

    # Replace spaces in column names with underscores
    df_raw_data.columns = df_raw_data.columns.str.replace(' ', '_')

    return df_raw_data

# Function to Define Variables
def _2_define_variables():
    pointwise_ranking_evaluation_columns = [
        'Data', 'Algorithm', 'Kendall_Tau', 'Spearman_Correlation', 'NDCG', 'Duration'
    ]

    nominal_columns = [
        'Genre', 'Age_Bin'
    ]

    ignore_columns = [
        'User_ID', 'User_Location', 'Book_Title', 'Publisher', 'Publication_Year'
    ]

    ordinal_columns = [
        'Rating1'
    ]
    return pointwise_ranking_evaluation_columns, nominal_columns, ordinal_columns, ignore_columns

# Function to split data
def _3_split_data(df_rawdata):
    X = df_rawdata[['Genre', 'Publisher', 'Publication_Year', 'Book_Title', 'User_Location', 'Age_Bin']]
    y = df_rawdata['Rating1']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test

# Function to preprocess data
def _4_preprocess_data(X_train, X_test, y_train, y_test, nominal_columns, ordinal_columns, ignore_columns):
    # Handle missing values in nominal columns
    for col in nominal_columns:
        X_train[col].fillna("Unknown", inplace=True)

    train_preprocessed_data = X_train.copy()
    test_preprocessed_data = X_test.copy()

    # Drop specified ignore columns
    if ignore_columns:
        train_preprocessed_data.drop(columns=ignore_columns, errors='ignore', inplace=True)
        test_preprocessed_data.drop(columns=ignore_columns, errors='ignore', inplace=True)

    # Add 'Rating1' back to the training set
    train_preprocessed_data['Rating1'] = y_train

    return train_preprocessed_data, test_preprocessed_data

# Function to Encode Data
def _5_encode_data(train_preprocessed, X_test_preprocessed, nominal_columns, ordinal_columns, ignore_columns):
    # One-hot encode nominal columns for training data
    train_preprocessed = pd.get_dummies(train_preprocessed, columns=nominal_columns, drop_first=True)

    # One-hot encode nominal columns for test data
    X_test_preprocessed = pd.get_dummies(X_test_preprocessed, columns=nominal_columns, drop_first=True)

    # Ensure that the columns in training and test data match after encoding
    common_columns = list(set(train_preprocessed.columns) & set(X_test_preprocessed.columns))
    train_preprocessed = train_preprocessed[common_columns]
    X_test_preprocessed = X_test_preprocessed[common_columns]

    # Drop columns specified in ignore_columns
    if ignore_columns:
        train_preprocessed.drop(columns=ignore_columns, errors='ignore', inplace=True)
        X_test_preprocessed.drop(columns=ignore_columns, errors='ignore', inplace=True)

    return train_preprocessed, X_test_preprocessed

# Function to evaluate pointwise ranking model
def evaluate_pointwise_ranking_model(data_name, result_df, clf_name, y_true, y_pred, duration):
    kendall_tau, _ = kendalltau(y_true, y_pred)
    spearman_corr, _ = spearmanr(y_true, y_pred)
    ndcg = ndcg_score([y_true], [y_pred])

    new_row = {
        'Data': data_name,
        'Algorithm': clf_name,
        'Duration': duration,
        'Kendall_Tau': kendall_tau,
        'Spearman_Correlation': spearman_corr,
        'NDCG': ndcg
    }

    result_df = pd.concat([result_df, pd.DataFrame([new_row])], ignore_index=True)
    return result_df

# Function to train pointwise ranking model
def train_pointwise_ranking_model(X_train, X_test, y_train, y_test, result_df):
    algorithms = {
        '1. XGBoost': XGBRegressor(),
        '2. Random_Forest': RandomForestRegressor()
    }

    for clf_name, clf in algorithms.items():
        # Fit the data and tag outliers
        t0 = time.time()
        model = clf
        model.fit(X_train, y_train)

        y_pred_train = model.predict(X_train)
        duration = round(time.time() - t0, 2)
        result_df = evaluate_pointwise_ranking_model("Train_Data", result_df, clf_name, y_train, y_pred_train, duration)

        y_pred_test = model.predict(X_test)
        duration = round(time.time() - t0, 2)
        result_df = evaluate_pointwise_ranking_model("Test_Data", result_df, clf_name, y_test, y_pred_test, duration)

    return result_df

# Main function
def main():
    # 0. Start the timer
    start_time = time.time()

    # Step 1. Get Training Data
    df_rawdata = _1_get_data()

    # Step 2. Define Variables
    pointwise_ranking_evaluation_columns, nominal_columns, ordinal_columns, ignore_columns = _2_define_variables()
    pointwise_ranking_result_df = pd.DataFrame(columns=pointwise_ranking_evaluation_columns)

    # Step 3. Split data
    X_train, X_test, y_train, y_test = _3_split_data(df_rawdata)

    # Step 4. Preprocess data
    X_train_preprocessed, X_test_preprocessed = _4_preprocess_data(X_train, X_test, y_train, y_test, nominal_columns, ordinal_columns, ignore_columns)

    # Step 5. Encode data
    train_encoded_data, test_encoded_data = _5_encode_data(X_train_preprocessed, X_test_preprocessed, nominal_columns, ordinal_columns, ignore_columns)

    # Step 8. Train and evaluate the model
    pointwise_ranking_result_df = train_pointwise_ranking_model(train_encoded_data, test_encoded_data, y_train, y_test, pointwise_ranking_result_df)

    return pointwise_ranking_result_df

if __name__ == "__main__":
    pointwise_ranking_result_df = main()
    pointwise_ranking_result_df


In [ ]:
# Print Header
print_pretty_header("Ranking Algorithm", "Pointwise")
# Print results
pointwise_ranking_result_df

####################################################################################################

                                         Ranking Algorithm                                          

                                             Pointwise                                              

####################################################################################################



,Data,Algorithm,Kendall_Tau,Spearman_Correlation,NDCG,Duration
0,Train_Data,1. XGBoost,0.057074,0.080282,0.934506,0.13
1,Test_Data,1. XGBoost,-0.025067,-0.035431,0.905422,0.15
2,Train_Data,2. Random_Forest,0.056799,0.079977,0.934494,0.45
3,Test_Data,2. Random_Forest,-0.024978,-0.035436,0.905300,0.48


------------- End of S3_Ch11_Ranking Chapter Code ------------ Vidya Subramanian ------------------